In [ ]:
import pandas as pd
from pathlib import Path
from datetime import datetime
from pydantic import BaseModel, model_validator
from typing import ClassVar

from src.data_preparation import ascat_era5 as utils
from src.data_preparation.general import filter_df, add_class_col
from src.data_preparation.validation import validate_time_index, check_df_cols
from src.data_preparation.visualization import plot
from src.constants import constants as c
from src.constants import StationName

# Data Understanding and Preparation of the ASCAT and ERA5 Data

The goal is to generate descriptive statistics, drop missing values & outliers, and plot the data for all ASCAT and ERA5 data in `raw_path / ASCAT` and `raw_path / ERA5`. The cleaned data will then be exported to their respective folders in `c.CLEANED_DATA_PATH`.

---

### Table of Contents

1. **Setup**
    - *1.1 Variables*
    - *1.2 Functions*
2. **ISMN Stations**
    - *2.1 Aberdeen-35-WNW*
    - *2.2 Jamestown-38-WSW*
    - *2.3 Gobblers Knob*
    - *2.4 Nenana*
    - *2.5 L23*
    - *2.6 L38*
    - *2.7 NST-07*
    - *2.8 NST-09*
    - *2.9 SOD012*
    - *2.10 SOD103*

---

## 1. Setup

### 1.1 Variables

In [ ]:
class NotebookVariables(BaseModel):
    """DO NOT EDIT THIS CLASS."""
    ASCAT_short_var_name: str = 'backscatter40'
    ERA5_short_var_name: str = 'stl1'
    ASCAT_long_var_name: str = '40° Normalized Backscatter (dB)'
    ERA5_long_var_name: str = 'Soil Temperature Level 1 (C)'

    ASCAT: ClassVar[str] = 'ASCAT'
    ERA5: ClassVar[str] = 'ERA5'

    raw_path: Path = Path('../data/raw/ASCAT_ERA5')

    @model_validator(mode="after")
    def validate(self):
        if not self.raw_path.is_dir():
            raise NotADirectoryError(f'{self.raw_path} must point to a directory.')
        return self

nb_vars = NotebookVariables()

### 1.2 Functions

In [ ]:
def data_cleaning(data_path: Path,
                  ismn_sites_path: Path,
                  station: str,
                  system: str,
                  key_variable: str,
                  class_boundary=None,
                  classes=None,
                  date_range=None) -> pd.DataFrame:
    """
    Preprocessing of raw ASCAT or ERA5 data.
    :param data_path: path to csv file
    :param ismn_sites_path: path to csv file containing ISMN site information
    :param station: must match exactly name in ISMN_site_survey.csv
    :param system: ASCAT or nb_vars.ERA5, case-insensitive
    :param key_variable: name of variable of interest (in csv)
    :param class_boundary: in C; creates a symmetric boundary across the freezing point for class labeling
    :param classes: list of exactly length three containing only str elements
    :param date_range: DateRange class. Data outside this range is filtered out. Start is inclusive and end is exclusive. Assumed that input is in the same timezone as df.index. If date_range is not valid for given df, date_range will be adjusted to be valid.
    :return: cleaned dataframe
    """
    # load data
    df = utils.collect_data(data_path, ismn_sites_path, station, system)
    check_df_cols(df, system)

    # report NaN count and proportion
    na_count = df.isnull().sum()[key_variable]
    print(f'There are {na_count} nulls out of {len(df)} datapoints ({round(na_count/len(df),2)}% missing).')

    # data preprocessing
    df = df.dropna(subset=[key_variable]) # no data imputation/interpolation for ASCAT
    df = utils.round_nearest_hour_index(df)
    if date_range is not None:
        df = filter_df(df, date_range.start, date_range.end)

    # only keep required columns
    if system.upper() == 'ASCAT':
        if c.ASCAT_KEY_COLS is not None:
            df = df[c.ASCAT_KEY_COLS]
    if system.upper() == 'ERA5':
        if c.ERA5_KEY_COLS is not None:
            df = df[c.ERA5_KEY_COLS]

    # data imputation and add predicted class column
    if system.upper() == 'ERA5':
        df = utils.impute_hourly(df)
        if class_boundary is not None and classes is not None:
            df = add_class_col(df, key_variable, 'pred')

    return df

def data_reporting(df: pd.DataFrame, variable: str, station_name: str, system: str, ylabel: str) -> None:
    """
    Reporting of preprocessed ASCAT and ERA5 data.
    :param df: can be output from soil_temp_data_cleaning()
    :param variable: full variable name
    :param station_name: official name of ISMN station
    :param system: name of sensor system
    :param ylabel: y-label for plot
    :return:
    """
    # check data types
    if not isinstance(df, pd.DataFrame):
        raise TypeError(f'df must be a pd.DataFrame, not {type(df)}.')
    # check input values
    if variable not in df.columns:
        raise KeyError(f'df missing required column "{variable}".')
    # check input df index
    validate_time_index(df)

    print('Summary statistics:')
    display(df.describe())
    print('Show head of df:')
    display(df.head())

    plot(df, variable, station=station_name, system=system, form='line', y_label=ylabel)

def plot_one_year(df: pd.DataFrame, variable: str, station: str, system: str, ylabel: str) -> None:
    """
    Plot one year of df from 2010-06-01 to 2011-06-01.
    :param df: df after all preprocessing
    :param variable: short variable name
    :param station: ISMN station name
    :param system: sensor system name
    :param ylabel: full variable name for plot
    :return: None
    """
    plot(df, variable, station, system, 'line', ylabel, start=datetime(2010, 6, 1), end=datetime(2011, 6, 1))

## 2. ISMN Stations

### 2.1 Aberdeen-35-WNW

#### ASCAT

In [ ]:
ASCAT_aberdeen_df = data_cleaning(data_path=nb_vars.raw_path,
                                  ismn_sites_path=c.SITE_SURVEY_PATH,
                                  station=StationName.ABERDEEN,
                                  system=nb_vars.ASCAT,
                                  key_variable=nb_vars.ASCAT_short_var_name,
                                  date_range=c.DATE_RANGE)
data_reporting(df=ASCAT_aberdeen_df,
               variable=nb_vars.ASCAT_short_var_name,
               station_name=StationName.ABERDEEN,
               system=nb_vars.ASCAT,
               ylabel=nb_vars.ASCAT_long_var_name)

Plot one year of ASCAT data to determine backscatter behavior for this location

In [ ]:
plot_one_year(ASCAT_aberdeen_df, nb_vars.ASCAT_short_var_name, StationName.ABERDEEN, nb_vars.ASCAT, nb_vars.ASCAT_long_var_name)

In [ ]:
# export to csv
ASCAT_aberdeen_df.to_csv(c.CLEANED_DATA_PATH / f'{StationName.ABERDEEN}_{nb_vars.ASCAT}.csv')

#### ERA5

In [ ]:
ERA5_aberdeen_df = data_cleaning(data_path=nb_vars.raw_path,
                                 ismn_sites_path=c.SITE_SURVEY_PATH,
                                 station=StationName.ABERDEEN,
                                 system=nb_vars.ERA5,
                                 key_variable=nb_vars.ERA5_short_var_name,
                                 class_boundary=c.CLASS_BOUNDARY,
                                 classes=c.CLASSES,
                                 date_range=c.DATE_RANGE)
data_reporting(df=ERA5_aberdeen_df,
               variable=nb_vars.ERA5_short_var_name,
               station_name=StationName.ABERDEEN,
               system=nb_vars.ERA5,
               ylabel=nb_vars.ERA5_long_var_name)

In [ ]:
# export to csv
ERA5_aberdeen_df.to_csv(c.CLEANED_DATA_PATH / f'{StationName.ABERDEEN}_{nb_vars.ERA5}.csv')

### 2.2 Jamestown-38-WSW

#### ASCAT

In [ ]:
ASCAT_jamestown_df = data_cleaning(data_path=nb_vars.raw_path,
                                   ismn_sites_path=c.SITE_SURVEY_PATH,
                                   station=StationName.JAMESTOWN,
                                   system=nb_vars.ASCAT,
                                   key_variable=nb_vars.ASCAT_short_var_name,
                                   date_range=c.DATE_RANGE)
data_reporting(df=ASCAT_jamestown_df,
               variable=nb_vars.ASCAT_short_var_name,
               station_name=StationName.JAMESTOWN,
               system=nb_vars.ASCAT,
               ylabel=nb_vars.ASCAT_long_var_name)

Plot one year of ASCAT data to determine backscatter behavior for this location

In [ ]:
plot_one_year(ASCAT_jamestown_df, nb_vars.ASCAT_short_var_name, StationName.JAMESTOWN, nb_vars.ASCAT, nb_vars.ASCAT_long_var_name)

In [ ]:
# export to csv
ASCAT_jamestown_df.to_csv(c.CLEANED_DATA_PATH / f'{StationName.JAMESTOWN}_{nb_vars.ASCAT}.csv')

#### ERA5

In [ ]:
ERA5_jamestown_df = data_cleaning(data_path=nb_vars.raw_path,
                                  ismn_sites_path=c.SITE_SURVEY_PATH,
                                  station=StationName.JAMESTOWN,
                                  system=nb_vars.ERA5,
                                  key_variable=nb_vars.ERA5_short_var_name,
                                  class_boundary=c.CLASS_BOUNDARY,
                                  classes=c.CLASSES,
                                  date_range=c.DATE_RANGE)
data_reporting(df=ERA5_jamestown_df,
               variable=nb_vars.ERA5_short_var_name,
               station_name=StationName.JAMESTOWN,
               system=nb_vars.ERA5,
               ylabel=nb_vars.ERA5_long_var_name)

In [ ]:
# export to csv
ERA5_jamestown_df.to_csv(c.CLEANED_DATA_PATH / f'{StationName.JAMESTOWN}_{nb_vars.ERA5}.csv')

### 2.3 Gobblers Knob

#### ASCAT

In [ ]:
ASCAT_gobblers_knob_df = data_cleaning(data_path=nb_vars.raw_path,
                                       ismn_sites_path=c.SITE_SURVEY_PATH,
                                       station=StationName.GOBBLERS_KNOB,
                                       system=nb_vars.ASCAT,
                                       key_variable=nb_vars.ASCAT_short_var_name,
                                       date_range=c.DATE_RANGE)
data_reporting(df=ASCAT_gobblers_knob_df,
               variable=nb_vars.ASCAT_short_var_name,
               station_name=StationName.GOBBLERS_KNOB,
               system=nb_vars.ASCAT,
               ylabel=nb_vars.ASCAT_long_var_name)

Plot one year of ASCAT data to determine backscatter behavior for this location

In [ ]:
plot_one_year(ASCAT_gobblers_knob_df, nb_vars.ASCAT_short_var_name, StationName.GOBBLERS_KNOB, nb_vars.ASCAT, nb_vars.ASCAT_long_var_name)

In [ ]:
# export to csv
ASCAT_gobblers_knob_df.to_csv(c.CLEANED_DATA_PATH / f'{StationName.GOBBLERS_KNOB}_{nb_vars.ASCAT}.csv')

#### ERA5

In [ ]:
ERA5_gobblers_knob_df = data_cleaning(data_path=nb_vars.raw_path,
                                      ismn_sites_path=c.SITE_SURVEY_PATH,
                                      station=StationName.GOBBLERS_KNOB,
                                      system=nb_vars.ERA5,
                                      key_variable=nb_vars.ERA5_short_var_name,
                                      class_boundary=c.CLASS_BOUNDARY,
                                      classes=c.CLASSES,
                                      date_range=c.DATE_RANGE)
data_reporting(df=ERA5_gobblers_knob_df,
               variable=nb_vars.ERA5_short_var_name,
               station_name=StationName.GOBBLERS_KNOB,
               system=nb_vars.ERA5,
               ylabel=nb_vars.ERA5_long_var_name)

In [ ]:
# export to csv
ERA5_gobblers_knob_df.to_csv(c.CLEANED_DATA_PATH / f'{StationName.GOBBLERS_KNOB}_{nb_vars.ERA5}.csv')

### 2.4 Nenana

#### ASCAT

In [ ]:
ASCAT_nenana_df = data_cleaning(data_path=nb_vars.raw_path,
                                ismn_sites_path=c.SITE_SURVEY_PATH,
                                station=StationName.NENANA,
                                system=nb_vars.ASCAT,
                                key_variable=nb_vars.ASCAT_short_var_name,
                                date_range=c.DATE_RANGE)
data_reporting(df=ASCAT_nenana_df,
               variable=nb_vars.ASCAT_short_var_name,
               station_name=StationName.NENANA,
               system=nb_vars.ASCAT,
               ylabel=nb_vars.ASCAT_long_var_name)

Plot one year of ASCAT data to determine backscatter behavior for this location

In [ ]:
plot_one_year(ASCAT_nenana_df, nb_vars.ASCAT_short_var_name, StationName.NENANA, nb_vars.ASCAT, nb_vars.ASCAT_long_var_name)

In [ ]:
# export to csv
ASCAT_nenana_df.to_csv(c.CLEANED_DATA_PATH / f'{StationName.NENANA}_{nb_vars.ASCAT}.csv')

#### ERA5

In [ ]:
ERA5_nenana_df = data_cleaning(data_path=nb_vars.raw_path,
                               ismn_sites_path=c.SITE_SURVEY_PATH,
                               station=StationName.NENANA,
                               system=nb_vars.ERA5,
                               key_variable=nb_vars.ERA5_short_var_name,
                               class_boundary=c.CLASS_BOUNDARY,
                               classes=c.CLASSES,
                               date_range=c.DATE_RANGE)
data_reporting(df=ERA5_nenana_df,
               variable=nb_vars.ERA5_short_var_name,
               station_name=StationName.NENANA,
               system=nb_vars.ERA5,
               ylabel=nb_vars.ERA5_long_var_name)

In [ ]:
# export to csv
ERA5_nenana_df.to_csv(c.CLEANED_DATA_PATH / f'{StationName.NENANA}_{nb_vars.ERA5}.csv')

### 2.5 L23

#### ASCAT

In [ ]:
ASCAT_L23_df = data_cleaning(data_path=nb_vars.raw_path,
                             ismn_sites_path=c.SITE_SURVEY_PATH,
                             station=StationName.L23,
                             system=nb_vars.ASCAT,
                             key_variable=nb_vars.ASCAT_short_var_name,
                             date_range=c.DATE_RANGE)
data_reporting(df=ASCAT_L23_df,
               variable=nb_vars.ASCAT_short_var_name,
               station_name=StationName.L23,
               system=nb_vars.ASCAT,
               ylabel=nb_vars.ASCAT_long_var_name)

Plot one year of ASCAT data to determine backscatter behavior for this location

In [ ]:
plot_one_year(ASCAT_L23_df, nb_vars.ASCAT_short_var_name, StationName.L23, nb_vars.ASCAT, nb_vars.ASCAT_long_var_name)

In [ ]:
# export to csv
ASCAT_L23_df.to_csv(c.CLEANED_DATA_PATH / f'{StationName.L23}_{nb_vars.ASCAT}.csv')

#### ERA5

In [ ]:
ERA5_L23_df = data_cleaning(data_path=nb_vars.raw_path,
                            ismn_sites_path=c.SITE_SURVEY_PATH,
                            station=StationName.L23,
                            system=nb_vars.ERA5,
                            key_variable=nb_vars.ERA5_short_var_name,
                            class_boundary=c.CLASS_BOUNDARY,
                            classes=c.CLASSES,
                            date_range=c.DATE_RANGE)
data_reporting(df=ERA5_L23_df,
               variable=nb_vars.ERA5_short_var_name,
               station_name=StationName.L23,
               system=nb_vars.ERA5,
               ylabel=nb_vars.ERA5_long_var_name)

In [ ]:
# export to csv
ERA5_L23_df.to_csv(c.CLEANED_DATA_PATH / f'{StationName.L23}_{nb_vars.ERA5}.csv')

### 2.6 L38

#### ASCAT

In [ ]:
ASCAT_L38_df = data_cleaning(data_path=nb_vars.raw_path,
                             ismn_sites_path=c.SITE_SURVEY_PATH,
                             station=StationName.L38,
                             system=nb_vars.ASCAT,
                             key_variable=nb_vars.ASCAT_short_var_name,
                             date_range=c.DATE_RANGE)
data_reporting(df=ASCAT_L38_df,
               variable=nb_vars.ASCAT_short_var_name,
               station_name=StationName.L38,
               system=nb_vars.ASCAT,
               ylabel=nb_vars.ASCAT_long_var_name)

Plot one year of ASCAT data to determine backscatter behavior for this location

In [ ]:
plot_one_year(ASCAT_L38_df, nb_vars.ASCAT_short_var_name, StationName.L38, nb_vars.ASCAT, nb_vars.ASCAT_long_var_name)

In [ ]:
# export to csv
ASCAT_L38_df.to_csv(c.CLEANED_DATA_PATH / f'{StationName.L38}_{nb_vars.ASCAT}.csv')

#### ERA5

In [ ]:
ERA5_L38_df = data_cleaning(data_path=nb_vars.raw_path,
                            ismn_sites_path=c.SITE_SURVEY_PATH,
                            station=StationName.L38,
                            system=nb_vars.ERA5,
                            key_variable=nb_vars.ERA5_short_var_name,
                            class_boundary=c.CLASS_BOUNDARY,
                            classes=c.CLASSES,
                            date_range=c.DATE_RANGE)
data_reporting(df=ERA5_L38_df,
               variable=nb_vars.ERA5_short_var_name,
               station_name=StationName.L38,
               system=nb_vars.ERA5,
               ylabel=nb_vars.ERA5_long_var_name)

In [ ]:
# export to csv
ERA5_L38_df.to_csv(c.CLEANED_DATA_PATH / f'{StationName.L38}_{nb_vars.ERA5}.csv')

### 2.7 NST-07

#### ASCAT

In [ ]:
ASCAT_NST_07_df = data_cleaning(data_path=nb_vars.raw_path,
                                ismn_sites_path=c.SITE_SURVEY_PATH,
                                station=StationName.NST_07,
                                system=nb_vars.ASCAT,
                                key_variable=nb_vars.ASCAT_short_var_name,
                                date_range=c.DATE_RANGE)
data_reporting(df=ASCAT_NST_07_df,
               variable=nb_vars.ASCAT_short_var_name,
               station_name=StationName.NST_07,
               system=nb_vars.ASCAT,
               ylabel=nb_vars.ASCAT_long_var_name)

Plot one year of ASCAT data to determine backscatter behavior for this location

In [ ]:
plot_one_year(ASCAT_NST_07_df, nb_vars.ASCAT_short_var_name, StationName.NST_07, nb_vars.ASCAT, nb_vars.ASCAT_long_var_name)

In [ ]:
# export to csv
ASCAT_NST_07_df.to_csv(c.CLEANED_DATA_PATH / f'{StationName.NST_07}_{nb_vars.ASCAT}.csv')

#### ERA5

In [ ]:
ERA5_NST_07_df = data_cleaning(data_path=nb_vars.raw_path,
                               ismn_sites_path=c.SITE_SURVEY_PATH,
                               station=StationName.NST_07,
                               system=nb_vars.ERA5,
                               key_variable=nb_vars.ERA5_short_var_name,
                               class_boundary=c.CLASS_BOUNDARY,
                               classes=c.CLASSES,
                               date_range=c.DATE_RANGE)
data_reporting(df=ERA5_NST_07_df,
               variable=nb_vars.ERA5_short_var_name,
               station_name=StationName.NST_07,
               system=nb_vars.ERA5,
               ylabel=nb_vars.ERA5_long_var_name)

In [ ]:
# export to csv
ERA5_NST_07_df.to_csv(c.CLEANED_DATA_PATH / f'{StationName.NST_07}_{nb_vars.ERA5}.csv')

### 2.8 NST-09

#### ASCAT

In [ ]:
ASCAT_NST_09_df = data_cleaning(data_path=nb_vars.raw_path,
                                ismn_sites_path=c.SITE_SURVEY_PATH,
                                station=StationName.NST_09,
                                system=nb_vars.ASCAT,
                                key_variable=nb_vars.ASCAT_short_var_name,
                                date_range=c.DATE_RANGE)
data_reporting(df=ASCAT_NST_09_df,
               variable=nb_vars.ASCAT_short_var_name,
               station_name=StationName.NST_09,
               system=nb_vars.ASCAT,
               ylabel=nb_vars.ASCAT_long_var_name)

Plot one year of ASCAT data to determine backscatter behavior for this location

In [ ]:
plot_one_year(ASCAT_NST_09_df, nb_vars.ASCAT_short_var_name, StationName.NST_09, nb_vars.ASCAT, nb_vars.ASCAT_long_var_name)

In [ ]:
# export to csv
ASCAT_NST_09_df.to_csv(c.CLEANED_DATA_PATH / f'{StationName.NST_09}_{nb_vars.ASCAT}.csv')

#### ERA5

In [ ]:
ERA5_NST_09_df = data_cleaning(data_path=nb_vars.raw_path,
                               ismn_sites_path=c.SITE_SURVEY_PATH,
                               station=StationName.NST_09,
                               system=nb_vars.ERA5,
                               key_variable=nb_vars.ERA5_short_var_name,
                               class_boundary=c.CLASS_BOUNDARY,
                               classes=c.CLASSES,
                               date_range=c.DATE_RANGE)
data_reporting(df=ERA5_NST_09_df,
               variable=nb_vars.ERA5_short_var_name,
               station_name=StationName.NST_09,
               system=nb_vars.ERA5,
               ylabel=nb_vars.ERA5_long_var_name)

In [ ]:
# export to csv
ERA5_NST_09_df.to_csv(c.CLEANED_DATA_PATH / f'{StationName.NST_09}_{nb_vars.ERA5}.csv')

### 2.9 SOD012

#### ASCAT

In [ ]:
ASCAT_SOD012_df = data_cleaning(data_path=nb_vars.raw_path,
                                ismn_sites_path=c.SITE_SURVEY_PATH,
                                station=StationName.SOD012,
                                system=nb_vars.ASCAT,
                                key_variable=nb_vars.ASCAT_short_var_name,
                                date_range=c.DATE_RANGE)
data_reporting(df=ASCAT_SOD012_df,
               variable=nb_vars.ASCAT_short_var_name,
               station_name=StationName.SOD012,
               system=nb_vars.ASCAT,
               ylabel=nb_vars.ASCAT_long_var_name)

Plot one year of ASCAT data to determine backscatter behavior for this location

In [ ]:
plot_one_year(ASCAT_SOD012_df, nb_vars.ASCAT_short_var_name, StationName.SOD012, nb_vars.ASCAT, nb_vars.ASCAT_long_var_name)

In [ ]:
# export to csv
ASCAT_SOD012_df.to_csv(c.CLEANED_DATA_PATH / f'{StationName.SOD012}_{nb_vars.ASCAT}.csv')

#### ERA5

In [ ]:
ERA5_SOD012_df = data_cleaning(data_path=nb_vars.raw_path,
                               ismn_sites_path=c.SITE_SURVEY_PATH,
                               station=StationName.SOD012,
                               system=nb_vars.ERA5,
                               key_variable=nb_vars.ERA5_short_var_name,
                               class_boundary=c.CLASS_BOUNDARY,
                               classes=c.CLASSES,
                               date_range=c.DATE_RANGE)
data_reporting(df=ERA5_SOD012_df,
               variable=nb_vars.ERA5_short_var_name,
               station_name=StationName.SOD012,
               system=nb_vars.ERA5,
               ylabel=nb_vars.ERA5_long_var_name)

In [ ]:
# export to csv
ERA5_SOD012_df.to_csv(c.CLEANED_DATA_PATH / f'{StationName.SOD012}_{nb_vars.ERA5}.csv')

### 2.10 SOD103

#### ASCAT

In [ ]:
ASCAT_SOD103_df = data_cleaning(data_path=nb_vars.raw_path,
                                ismn_sites_path=c.SITE_SURVEY_PATH,
                                station=StationName.SOD103,
                                system=nb_vars.ASCAT,
                                key_variable=nb_vars.ASCAT_short_var_name,
                                date_range=c.DATE_RANGE)
data_reporting(df=ASCAT_SOD103_df,
               variable=nb_vars.ASCAT_short_var_name,
               station_name=StationName.SOD103,
               system=nb_vars.ASCAT,
               ylabel=nb_vars.ASCAT_long_var_name)

Plot one year of ASCAT data to determine backscatter behavior for this location

In [ ]:
plot_one_year(ASCAT_SOD103_df, nb_vars.ASCAT_short_var_name, StationName.SOD103, nb_vars.ASCAT, nb_vars.ASCAT_long_var_name)

In [ ]:
# export to csv
ASCAT_SOD103_df.to_csv(c.CLEANED_DATA_PATH / f'{StationName.SOD103}_{nb_vars.ASCAT}.csv')

#### ERA5

In [ ]:
ERA5_SOD103_df = data_cleaning(data_path=nb_vars.raw_path,
                               ismn_sites_path=c.SITE_SURVEY_PATH,
                               station=StationName.SOD103,
                               system=nb_vars.ERA5,
                               key_variable=nb_vars.ERA5_short_var_name,
                               class_boundary=c.CLASS_BOUNDARY,
                               classes=c.CLASSES,
                               date_range=c.DATE_RANGE)
data_reporting(df=ERA5_SOD103_df,
               variable=nb_vars.ERA5_short_var_name,
               station_name=StationName.SOD103,
               system=nb_vars.ERA5,
               ylabel=nb_vars.ERA5_long_var_name)

In [ ]:
# export to csv
ERA5_SOD103_df.to_csv(c.CLEANED_DATA_PATH / f'{StationName.SOD103}_{nb_vars.ERA5}.csv')